# 💳 Credit Card Fraud Detection & Risk Monitoring
Kaggle `creditcard.csv` · scikit-learn · imbalanced classification · Google Colab

**How to use:** run the cells from top to bottom (Runtime → Run all, or Shift+Enter one at a time).
Read the markdown above each section, then the output below it.

| Part | What it does |
|---|---|
| 1 | Setup |
| 2 | Load the data |
| 3 | EDA |
| 4 | Preprocessing and splits |
| 5 | Models and the 11 experiments |
| 6 | Compare models (PR and ROC curves) |
| 7 | Threshold analysis |
| 8 | Business simulation and top-K |
| 9 | Risk levels and fraud alert list |
| 10 | Interactive dashboard |
| 11 | Final model, test evaluation (once) |
| 12 | Time-based split check (more realistic) |
| 13 | Save, and build the Streamlit app for deployment |

**The big idea of this project:** only about 0.17% of transactions are fraud, so *accuracy is useless*.
A model that says "legitimate" every time scores about 99.83% and catches zero fraud.
We judge models with recall, precision, PR-AUC, and by what they would save a bank.

## 1. Setup

In [ ]:
import os, sys, json, time, shutil, warnings
sys.path.insert(0, os.getcwd())

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from IPython.display import display, clear_output

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             average_precision_score, roc_auc_score, confusion_matrix,
                             precision_recall_curve, roc_curve, ConfusionMatrixDisplay)

try:
    from imblearn.over_sampling import SMOTE
except ImportError:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "imbalanced-learn"])
    from imblearn.over_sampling import SMOTE

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

# ---- SETTINGS YOU CAN CHANGE ----
FAST_MODE = True      # True = smaller forests, quick first run. Set False for the full-size models (slower).
RF_TREES = 40 if FAST_MODE else 100

COL = {0: "#2a9d8f", 1: "#e63946"}   # legitimate = green, fraud = red
print("xgboost available:", HAS_XGB, "| FAST_MODE:", FAST_MODE, "| RF trees:", RF_TREES)

## 2. Load the data
Tries three ways in order: a `creditcard.csv` already in Colab's file panel → automatic Kaggle download →
an upload button (download the file from Kaggle yourself first, about 150 MB).

In [ ]:
DATA_PATH = "creditcard.csv" if os.path.exists("creditcard.csv") else None

if DATA_PATH is None:
    try:
        try:
            import kagglehub
        except ImportError:
            import subprocess
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kagglehub"])
            import kagglehub
        folder = kagglehub.dataset_download("mlg-ulb/creditcardfraud")
        DATA_PATH = os.path.join(folder, "creditcard.csv")
    except Exception as e:
        print("Automatic Kaggle download failed:", e)

if DATA_PATH is None or not os.path.exists(DATA_PATH):
    from google.colab import files
    print("Please choose creditcard.csv")
    uploaded = files.upload()
    DATA_PATH = list(uploaded.keys())[0]

df = pd.read_csv(DATA_PATH)
print("Loaded:", DATA_PATH, "| shape:", df.shape)

## 3. EDA
Questions we answer in this section:
1. How many legitimate vs fraud transactions? 2. What percent is fraud? 3. Do fraud amounts differ?
4. Which features separate fraud? 5. How does time relate to fraud? 6. What if we predict "legitimate" for everything?

### 3.1 First look, missing values, duplicates

In [ ]:
display(df.head())
print()
df.info()
print("\nMissing values:", int(df.isnull().sum().sum()))
print("Duplicate rows:", int(df.duplicated().sum()))
display(df[["Time", "Amount", "Class"]].describe().T.round(3))

### 3.2 Class balance — the most important chart

In [ ]:
counts = df["Class"].value_counts().sort_index()
n, n_fraud = len(df), int(counts.get(1, 0))
print(f"Legitimate: {counts[0]:,}")
print(f"Fraud     : {n_fraud:,}")
print(f"Fraud rate: {n_fraud / n * 100:.3f}%   (about 1 fraud in {n // max(n_fraud,1):,} transactions)")

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].bar(["Legitimate", "Fraud"], counts.values, color=[COL[0], COL[1]])
ax[0].set_yscale("log"); ax[0].set_title("Transactions per class (log scale!)")
for i, v in enumerate(counts.values):
    ax[0].text(i, v, f"{v:,}", ha="center", va="bottom", fontweight="bold")
ax[1].pie(counts.values, labels=["Legitimate", "Fraud"], colors=[COL[0], COL[1]],
          autopct="%1.3f%%", startangle=90, explode=(0, 0.15))
ax[1].set_title("Share of transactions")
plt.tight_layout(); plt.show()

# What if we predict "legitimate" for everything?
naive_acc = counts[0] / n
print(f"\nPredict 'legitimate' for EVERY transaction -> accuracy = {naive_acc*100:.2f}%, fraud caught = 0")
print("That is why accuracy is useless in this project.")

### 3.3 Transaction amount

In [ ]:
print("Amount summary by class:")
display(df.groupby("Class")["Amount"].describe().round(2))

fig, ax = plt.subplots(1, 3, figsize=(18, 4.5))
for c, lab in [(0, "Legitimate"), (1, "Fraud")]:
    sns.histplot(np.log1p(df.loc[df["Class"] == c, "Amount"]), bins=50, stat="density",
                 color=COL[c], alpha=0.5, label=lab, ax=ax[0])
ax[0].set_title("log(1 + Amount), normalised"); ax[0].set_xlabel("log1p(Amount)"); ax[0].legend()

sns.boxplot(data=df.assign(logAmount=np.log1p(df["Amount"])), x="Class", y="logAmount",
            palette=[COL[0], COL[1]], ax=ax[1])
ax[1].set_title("Amount by class (box plot, log scale)"); ax[1].set_xticklabels(["Legitimate", "Fraud"])

sns.histplot(df.loc[df["Class"] == 1, "Amount"], bins=40, color=COL[1], ax=ax[2])
ax[2].set_title("Fraud amounts (raw $)")
plt.tight_layout(); plt.show()
print(f"Median amount  legit: ${df.loc[df.Class==0,'Amount'].median():.2f}   fraud: ${df.loc[df.Class==1,'Amount'].median():.2f}")
print(f"Total $ in fraud transactions: ${df.loc[df.Class==1,'Amount'].sum():,.0f}")

### 3.4 Time
`Time` is seconds since the first transaction (the data covers two days), so `(Time // 3600) % 24` is roughly the hour of day.

In [ ]:
hour = ((df["Time"] // 3600) % 24).astype(int)
fig, ax = plt.subplots(1, 3, figsize=(18, 4.5))
ax[0].hist(df["Time"] / 3600, bins=48, color="#457b9d"); ax[0].set_title("All transactions over the 2 days"); ax[0].set_xlabel("hours since start")
for c, lab in [(0, "Legitimate"), (1, "Fraud")]:
    ax[1].hist(hour[df["Class"] == c], bins=24, range=(0, 24), density=True, alpha=0.55, color=COL[c], label=lab)
ax[1].set_title("Hour of day (normalised)"); ax[1].set_xlabel("hour"); ax[1].legend()
rate = df.groupby(hour)["Class"].mean() * 100
ax[2].bar(rate.index, rate.values, color=COL[1]); ax[2].set_title("Fraud rate (%) by hour of day"); ax[2].set_xlabel("hour")
plt.tight_layout(); plt.show()
print("Fraud rate is highest around hours:", list(rate.sort_values(ascending=False).head(3).index))

### 3.5 The V1–V28 features

In [ ]:
V_COLS = [f"V{i}" for i in range(1, 29)]
corr_t = df[V_COLS + ["Amount"]].corrwith(df["Class"]).sort_values()
plt.figure(figsize=(8, 9))
plt.barh(corr_t.index, corr_t.values, color=[COL[1] if v > 0 else "#457b9d" for v in corr_t.values])
plt.title("Correlation of each feature with Class (fraud = 1)"); plt.xlabel("correlation")
plt.tight_layout(); plt.show()

top = corr_t.abs().sort_values(ascending=False).head(8).index.tolist()
print("Most fraud-related features:", top)

fig, axes = plt.subplots(2, 4, figsize=(20, 7))
for ax, col in zip(axes.ravel(), top):
    for c, lab in [(0, "Legitimate"), (1, "Fraud")]:
        sns.kdeplot(df.loc[df["Class"] == c, col], color=COL[c], fill=True, alpha=0.4, label=lab, ax=ax, common_norm=False)
    ax.set_title(col); ax.legend()
plt.suptitle("Where fraud and legitimate transactions separate", y=1.02, fontsize=14)
plt.tight_layout(); plt.show()

In [ ]:
cols12 = corr_t.abs().sort_values(ascending=False).head(12).index.tolist()
plt.figure(figsize=(10, 8))
sns.heatmap(df[cols12 + ["Class"]].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True)
plt.title("Correlation heatmap: 12 most fraud-related features + Class")
plt.tight_layout(); plt.show()
print("V1-V28 are PCA components, so they are almost uncorrelated with each other by construction.")

### 3.6 EDA summary — write your own answers
1. Legitimate transactions: ...  2. Fraud transactions: ...  3. Fraud rate: ...
4. Do fraud amounts differ? ...  5. Best separating features: ...  6. Time pattern: ...
7. Accuracy of "always legitimate": ...

## 4. Preprocessing and splits

| Column | Treatment |
|---|---|
| `Time` | replaced by `Hour` (hour of day), the raw seconds counter is dropped |
| `V1`–`V28` | used as provided |
| `Amount` | `log1p` (very skewed) then standardised |
| `Class` | target, never an input |

The feature logic lives in a small file, `fraud_utils.py`, written by the next cell. The Streamlit app reuses the exact same code,
so training and deployment can never drift apart.

Split: **stratified 60 / 20 / 20** (train / validation / test), keeping the fraud share equal in each part.
The scaler is fitted on the training set only.

In [ ]:
FRAUD_UTILS_SRC = """
import numpy as np
import pandas as pd

V_COLS = [f"V{i}" for i in range(1, 29)]
SCALE_COLS = ["Amount", "Hour"]
FEATURES = V_COLS + ["Amount", "Hour"]
RISK_BINS = [0.0, 0.2, 0.5, 0.8, 1.0]
RISK_LABELS = ["Low", "Medium", "High", "Critical"]
REQUIRED_COLUMNS = ["Time", "Amount"] + V_COLS


def validate(df):
    # returns a list of problems (empty list = OK)
    problems = []
    missing = [c for c in REQUIRED_COLUMNS if c not in df.columns]
    if missing:
        problems.append("Missing columns: " + ", ".join(missing))
    else:
        n_na = int(df[REQUIRED_COLUMNS].isnull().sum().sum())
        if n_na:
            problems.append(f"{n_na} missing values in the feature columns")
    return problems


def add_features(df):
    # Time -> Hour of day, Amount -> log1p. Keeps Class if present.
    out = df.copy()
    out["Hour"] = ((out["Time"] // 3600) % 24).astype(float)
    out["Amount"] = np.log1p(out["Amount"])
    return out


def prepare(df, scaler):
    # raw transactions -> model-ready feature matrix, using a fitted scaler
    X = add_features(df)[FEATURES].copy()
    X[SCALE_COLS] = scaler.transform(X[SCALE_COLS])
    return X


def risk_level(proba):
    return pd.cut(pd.Series(np.asarray(proba, dtype=float)), bins=RISK_BINS,
                  labels=RISK_LABELS, include_lowest=True)
"""
with open("fraud_utils.py", "w") as f:
    f.write(FRAUD_UTILS_SRC)

import importlib, fraud_utils
importlib.reload(fraud_utils)
from fraud_utils import (V_COLS, SCALE_COLS, FEATURES, RISK_LABELS, add_features, prepare, risk_level, validate)
print("fraud_utils.py written and imported. Problems with the data:", validate(df) or "none")

In [ ]:
feat = add_features(df)
X = feat[FEATURES]
y = df["Class"]
AMOUNT_RAW = df["Amount"]            # original dollars, used by the business simulation

# stratified 60 / 20 / 20
X_dev, X_test, y_dev, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(X_dev, y_dev, test_size=0.25, stratify=y_dev, random_state=SEED)

def scale_with(scaler, Xdf):
    out = Xdf.copy()
    out[SCALE_COLS] = scaler.transform(out[SCALE_COLS])
    return out

scaler = StandardScaler().fit(X_train[SCALE_COLS])        # TRAIN only
X_train_s, X_val_s, X_test_s = (scale_with(scaler, d) for d in (X_train, X_val, X_test))

summary = pd.DataFrame({"transactions": [len(y_train), len(y_val), len(y_test)],
                        "fraud": [int(y_train.sum()), int(y_val.sum()), int(y_test.sum())]},
                       index=["train", "validation", "test"])
summary["fraud %"] = (summary["fraud"] / summary["transactions"] * 100).round(3)
display(summary)
print("\nScaled train Amount/Hour mean (about 0):", X_train_s[SCALE_COLS].mean().round(3).to_dict())

## 5. Models and experiments
**Metrics (fraud = positive class):**
- **Recall:** of all real frauds, how many did we catch?
- **Precision:** of all alerts, how many were real fraud?
- **F1:** balance of the two.
- **PR-AUC:** summary of the precision-recall curve, the main metric for rare positives.
- **ROC-AUC:** how well fraud is ranked above legitimate.
- Accuracy is shown only so you can see how misleading it is.

**Imbalance methods:** class weights (fraud errors cost more), **SMOTE** (synthetic fraud examples, training data only;
here fraud is raised to 10% of the legitimate count to keep training fast) and threshold changes (section 7).

In [ ]:
def make_model(kind, class_weight=False):
    cw = "balanced" if class_weight else None
    if kind == "dummy":  return DummyClassifier(strategy="most_frequent")
    if kind == "logreg": return LogisticRegression(max_iter=2000, class_weight=cw)
    if kind == "tree":   return DecisionTreeClassifier(max_depth=8, min_samples_leaf=5, class_weight=cw, random_state=SEED)
    if kind == "rf":     return RandomForestClassifier(n_estimators=RF_TREES, max_depth=12, min_samples_leaf=2, n_jobs=-1,
                                                       class_weight=("balanced_subsample" if class_weight else None), random_state=SEED)
    if kind == "xgb":
        spw = float((y_train == 0).sum() / max((y_train == 1).sum(), 1)) if class_weight else 1.0
        return XGBClassifier(n_estimators=150 if FAST_MODE else 300, max_depth=5, learning_rate=0.1, subsample=0.8,
                             scale_pos_weight=spw, eval_metric="aucpr", n_jobs=-1, random_state=SEED)
    raise ValueError(kind)

def fit_model(kind, class_weight, smote, Xtr, ytr):
    """Train one model. SMOTE (if on) is applied to the TRAINING data only."""
    if smote:
        Xtr, ytr = SMOTE(sampling_strategy=0.1, random_state=SEED).fit_resample(Xtr, ytr)
    model = make_model(kind, class_weight)
    model.fit(Xtr, ytr)
    return model

def score(y_true, proba, thr=0.5):
    pred = (np.asarray(proba) >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    return dict(
        accuracy=accuracy_score(y_true, pred),
        precision=precision_score(y_true, pred, zero_division=0),
        recall=recall_score(y_true, pred, zero_division=0),
        f1=f1_score(y_true, pred, zero_division=0),
        pr_auc=average_precision_score(y_true, proba),
        roc_auc=roc_auc_score(y_true, proba),
        tp=int(tp), fn=int(fn), fp=int(fp), tn=int(tn),
    )

print("Helpers ready.")

In [ ]:
RESULTS, PROBAS, CONFIGS, MODELS = [], {}, {}, {}

def log_row(name, model_label, technique, thr, s, seconds=0.0):
    RESULTS.append({
        "Experiment": name, "Model": model_label, "Technique": technique, "Threshold": thr,
        "Accuracy": round(s["accuracy"], 4), "Precision": round(s["precision"], 4),
        "Recall": round(s["recall"], 4), "F1": round(s["f1"], 4),
        "PR-AUC": round(s["pr_auc"], 4), "ROC-AUC": round(s["roc_auc"], 4),
        "Fraud Caught": s["tp"], "Fraud Missed": s["fn"], "False Alarms": s["fp"],
        "Train Time (s)": round(seconds, 1),
    })

def run_experiment(name, kind, label, technique, class_weight=False, smote=False):
    t0 = time.time()
    model = fit_model(kind, class_weight, smote, X_train_s, y_train)
    proba = model.predict_proba(X_val_s)[:, 1]
    log_row(name, label, technique, 0.5, score(y_val, proba), time.time() - t0)
    PROBAS[name], CONFIGS[name], MODELS[name] = proba, (kind, class_weight, smote), model
    r = RESULTS[-1]
    print(f"{name:<34} recall={r['Recall']:.3f} precision={r['Precision']:.3f} PR-AUC={r['PR-AUC']:.3f}  ({r['Train Time (s)']}s)")

EXPERIMENT_PLAN = [
    ("1. Dummy",                  "dummy",  "Dummy (majority class)", "None", False, False),
    ("2. LogReg default",         "logreg", "Logistic Regression",    "Default", False, False),
    ("3. LogReg class weights",   "logreg", "Logistic Regression",    "Class weights", True, False),
    ("4. Decision Tree",          "tree",   "Decision Tree",          "Default", False, False),
    ("5. Random Forest",          "rf",     "Random Forest",          "Default", False, False),
    ("6. RF class weights",       "rf",     "Random Forest",          "Class weights", True, False),
    ("7. RF SMOTE",               "rf",     "Random Forest",          "SMOTE", False, True),
]
if HAS_XGB:
    EXPERIMENT_PLAN.append(("8. XGBoost (optional)", "xgb", "XGBoost", "Default", False, False))

for args in EXPERIMENT_PLAN:
    run_experiment(*args)

In [ ]:
# Experiments 9-11: the BEST model at three thresholds (no retraining, only the cut-off changes).
# "Best" = highest validation PR-AUC, ignoring the dummy.
res = pd.DataFrame(RESULTS)
best_name = res[res["Model"] != "Dummy (majority class)"].sort_values(["PR-AUC", "Recall"], ascending=False).iloc[0]["Experiment"]
print("Best model by validation PR-AUC:", best_name)

for i, thr in zip((9, 10, 11), (0.1, 0.3, 0.5)):
    s = score(y_val, PROBAS[best_name], thr)
    log_row(f"{i}. Best @ threshold {thr}", CONFIGS[best_name][0], f"Threshold {thr}", thr, s)

results_df = pd.DataFrame(RESULTS)
display(results_df)

**Read the table like this:** look at the Dummy row. Its accuracy is about 99.8% but *Fraud Caught = 0*. Then compare Recall and PR-AUC
across the other rows, and see how lowering the threshold in rows 9–11 trades more caught fraud for more false alarms.

## 6. Compare models (validation set)

In [ ]:
cmp = results_df[results_df["Experiment"].str[:2].str.strip(". ").astype(int) <= len(EXPERIMENT_PLAN)].set_index("Experiment")
fig, ax = plt.subplots(1, 2, figsize=(16, 5))
cmp[["Precision", "Recall", "F1", "PR-AUC"]].plot(kind="bar", ax=ax[0], colormap="viridis", edgecolor="white")
ax[0].set_title("Fraud metrics at threshold 0.5"); ax[0].set_ylim(0, 1.05); ax[0].tick_params(axis="x", rotation=40)
cmp[["Accuracy"]].plot(kind="bar", ax=ax[1], color="#999999", legend=False)
ax[1].set_ylim(0.99, 1.0005); ax[1].set_title("Accuracy (zoomed in: all models look 'great')"); ax[1].tick_params(axis="x", rotation=40)
plt.tight_layout(); plt.show()

In [ ]:
names_ex_dummy = [n for n in PROBAS if not n.startswith("1.")]
fig, ax = plt.subplots(1, 2, figsize=(15, 6))
for n in names_ex_dummy:
    p, r, _ = precision_recall_curve(y_val, PROBAS[n])
    ax[0].plot(r, p, label=f"{n} (AP={average_precision_score(y_val, PROBAS[n]):.2f})", lw=2)
    fpr, tpr, _ = roc_curve(y_val, PROBAS[n])
    ax[1].plot(fpr, tpr, label=n, lw=2)
ax[0].axhline(y_val.mean(), color="grey", ls="--", label="random (fraud rate)")
ax[0].set_xlabel("Recall"); ax[0].set_ylabel("Precision"); ax[0].set_title("Precision-Recall curves"); ax[0].legend(fontsize=8)
ax[1].plot([0, 1], [0, 1], "k--", lw=1); ax[1].set_xlabel("False positive rate"); ax[1].set_ylabel("True positive rate")
ax[1].set_title("ROC curves"); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# Confusion matrices at threshold 0.5 for the main models
show = [n for n in ("2. LogReg default", "3. LogReg class weights", "5. Random Forest", "7. RF SMOTE") if n in PROBAS]
fig, axes = plt.subplots(1, len(show), figsize=(4.8 * len(show), 4.2))
axes = np.atleast_1d(axes)
for ax, n in zip(axes, show):
    cm = confusion_matrix(y_val, (PROBAS[n] >= 0.5).astype(int), labels=[0, 1])
    ConfusionMatrixDisplay(cm, display_labels=["Legit", "Fraud"]).plot(ax=ax, cmap="Blues", values_format="d", colorbar=False)
    ax.set_title(n, fontsize=10)
plt.tight_layout(); plt.show()

**Questions for your report:**
1. What does the Dummy model prove about accuracy?
2. Did class weights improve recall for Logistic Regression? What did it cost in precision / false alarms?
3. Random Forest vs Logistic Regression: which has the better PR-AUC?
4. Did SMOTE help the forest, or are class weights / the default better?
5. Which model would you deploy, and why?

## 7. Threshold analysis
The model outputs a fraud *probability*. The threshold decides what counts as an alert.
Lower threshold → more fraud caught, more false alarms. We study this on the best model, using validation data.

In [ ]:
best_proba = PROBAS[best_name]
grid = np.round(np.arange(0.05, 0.96, 0.05), 2)
th = pd.DataFrame([{"Threshold": t, **{k: round(v, 4) for k, v in score(y_val, best_proba, t).items()
                                     if k in ("precision", "recall", "f1", "tp", "fn", "fp")}} for t in grid])
th = th.rename(columns={"precision": "Precision", "recall": "Recall", "f1": "F1", "tp": "Fraud Caught", "fn": "Fraud Missed", "fp": "False Alarms"})
display(th)

fig, ax = plt.subplots(1, 2, figsize=(15, 5))
ax[0].plot(th["Threshold"], th["Precision"], "o-", label="Precision"); ax[0].plot(th["Threshold"], th["Recall"], "o-", label="Recall")
ax[0].plot(th["Threshold"], th["F1"], "o-", label="F1", color="black")
ax[0].set_xlabel("threshold"); ax[0].set_title(f"Metrics vs threshold ({best_name})"); ax[0].legend()
ax[1].plot(th["Threshold"], th["False Alarms"], "o-", color="#e76f51", label="False alarms")
ax[1].plot(th["Threshold"], th["Fraud Missed"], "o-", color="#264653", label="Fraud missed")
ax[1].set_xlabel("threshold"); ax[1].set_title("The trade-off in counts"); ax[1].legend()
plt.tight_layout(); plt.show()

## 8. Business simulation
These are **assumptions you can change**, not real bank figures.
- average loss of a fraud we fail to stop: `AVG_FRAUD_LOSS`
- cost of investigating one alert: `ALERT_COST`
- share of detected frauds we manage to actually prevent: `PREVENT_RATE`

In [ ]:
BUSINESS = dict(avg_fraud_loss=200.0, alert_cost=5.0, prevent_rate=0.80)

def business(y_true, proba, thr, params=BUSINESS):
    pred = (np.asarray(proba) >= thr)
    yt = np.asarray(y_true).astype(bool)
    caught, missed = int((pred & yt).sum()), int((~pred & yt).sum())
    flagged = int(pred.sum())
    prevented = caught * params["avg_fraud_loss"] * params["prevent_rate"]
    cost = flagged * params["alert_cost"]
    no_model_loss = int(yt.sum()) * params["avg_fraud_loss"]
    return {"Alerts raised": flagged, "Fraud detected": caught, "Fraud missed": missed,
            "Losses prevented ($)": prevented, "Investigation cost ($)": cost,
            "Net benefit ($)": prevented - cost, "Loss with no model ($)": no_model_loss}

rows = []
for t in (0.1, 0.2, 0.3, 0.5, 0.7):
    rows.append({"Threshold": t, **business(y_val, best_proba, t)})
biz = pd.DataFrame(rows).set_index("Threshold")
display(biz.round(0))
t_best_biz = biz["Net benefit ($)"].idxmax()
print(f"Highest net benefit among these thresholds: {t_best_biz}  -> best business threshold may differ from best F1 threshold.")

In [ ]:
# Top-K prioritisation: if analysts can only check the K highest-risk transactions, how many frauds do we find?
order = np.argsort(-best_proba)
yv = y_val.values
rows = []
for K in (50, 100, 200, 500):
    hit = int(yv[order[:K]].sum())
    rows.append({"K (transactions checked)": K, "Frauds found": hit, "Precision@K": round(hit / K, 3),
                 "Share of all fraud found": round(hit / max(yv.sum(), 1), 3)})
display(pd.DataFrame(rows))

## 9. Risk levels and the fraud alert list
| Fraud probability | Risk |
|---|---|
| 0–20% | Low |
| 20–50% | Medium |
| 50–80% | High |
| 80–100% | Critical |

These are dashboard labels, not real bank policy.

In [ ]:
def alert_table(idx, proba, y_true=None):
    t = pd.DataFrame({"Transaction": idx, "Amount ($)": AMOUNT_RAW.loc[idx].round(2).values,
                      "Fraud probability": np.round(np.asarray(proba, dtype=float), 4)})
    t["Risk"] = risk_level(t["Fraud probability"]).astype(str).values
    if y_true is not None:
        t["Actually fraud"] = np.where(np.asarray(y_true) == 1, "YES", "no")
    return t.sort_values("Fraud probability", ascending=False).reset_index(drop=True)

alerts_val = alert_table(X_val.index, best_proba, y_val)
print("Risk level counts on the validation set:")
display(alerts_val["Risk"].value_counts().reindex(RISK_LABELS).fillna(0).astype(int).to_frame("transactions"))
print("\nTop 15 highest-risk transactions:")
display(alerts_val.head(15))

## 10. Interactive dashboard
Pick a model and imbalance method, press **Train**, then move the **threshold** slider: the metrics, confusion matrix and alert list update instantly
(no retraining). Results here are on the **validation** set. The test set is saved for section 11.
Filters (risk level, minimum probability, amount range) apply to the alert list.

In [ ]:
import ipywidgets as widgets

KIND = {"Logistic Regression": "logreg", "Decision Tree": "tree", "Random Forest": "rf"}
if HAS_XGB: KIND["XGBoost"] = "xgb"

sty = {"description_width": "130px"}; lay = widgets.Layout(width="460px")
w_model = widgets.Dropdown(options=list(KIND), value="Logistic Regression", description="Model", style=sty, layout=lay)
w_cw    = widgets.Checkbox(value=True, description="Class weights", style=sty)
w_smote = widgets.Checkbox(value=False, description="SMOTE (training data only)", style=sty)
w_thr   = widgets.FloatSlider(value=0.5, min=0.01, max=0.99, step=0.01, description="Threshold", style=sty, layout=lay, continuous_update=False)
w_risk  = widgets.SelectMultiple(options=RISK_LABELS, value=tuple(RISK_LABELS), description="Risk levels", style=sty, layout=widgets.Layout(width="460px", height="90px"))
w_minp  = widgets.FloatSlider(value=0.0, min=0.0, max=1.0, step=0.05, description="Min probability", style=sty, layout=lay, continuous_update=False)
amax = float(np.ceil(AMOUNT_RAW.max()))
w_amt   = widgets.FloatRangeSlider(value=(0, amax), min=0, max=amax, step=1, description="Amount range ($)", style=sty, layout=lay, continuous_update=False)
btn     = widgets.Button(description="Train", button_style="success", icon="play")
out     = widgets.Output()
DASH = {"proba": None, "label": ""}

def render(_=None):
    if DASH["proba"] is None:
        return
    proba, thr = DASH["proba"], w_thr.value
    s = score(y_val, proba, thr)
    b = business(y_val, proba, thr)
    with out:
        clear_output(wait=True)
        print(f"💳 Fraud Lab — {DASH['label']}  |  threshold {thr:.2f}  |  validation set")
        print("-" * 62)
        print(f"Fraud caught : {s['tp']} / {s['tp'] + s['fn']}      Recall   : {s['recall']*100:.1f}%")
        print(f"False alarms : {s['fp']}                 Precision: {s['precision']*100:.1f}%")
        print(f"F1           : {s['f1']:.3f}              PR-AUC   : {s['pr_auc']:.3f}    ROC-AUC: {s['roc_auc']:.3f}")
        print(f"Net benefit  : ${b['Net benefit ($)']:,.0f}   (prevented ${b['Losses prevented ($)']:,.0f} - investigation ${b['Investigation cost ($)']:,.0f})")
        fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
        ConfusionMatrixDisplay(confusion_matrix(y_val, (proba >= thr).astype(int), labels=[0, 1]),
                               display_labels=["Legit", "Fraud"]).plot(ax=ax[0], cmap="Blues", values_format="d", colorbar=False)
        ax[0].set_title("Confusion matrix")
        p, r, _ = precision_recall_curve(y_val, proba)
        ax[1].plot(r, p, lw=2); ax[1].scatter([s["recall"]], [s["precision"]], color="red", zorder=5, label="current threshold")
        ax[1].set_xlabel("Recall"); ax[1].set_ylabel("Precision"); ax[1].set_title("Precision-recall curve"); ax[1].legend()
        plt.tight_layout(); plt.show()

        a = alert_table(X_val.index, proba, y_val)
        a = a[(a["Fraud probability"] >= max(thr, w_minp.value)) & a["Risk"].isin(list(w_risk.value))
              & a["Amount ($)"].between(*w_amt.value)]
        print(f"Alert list: {len(a)} transactions flagged after filters (top 15 shown)")
        display(a.head(15))

def on_train(_):
    btn.disabled = True; btn.description = "Training..."
    try:
        kind = KIND[w_model.value]
        model = fit_model(kind, w_cw.value, w_smote.value, X_train_s, y_train)
        DASH["proba"] = model.predict_proba(X_val_s)[:, 1]
        DASH["label"] = f"{w_model.value}{' + class weights' if w_cw.value else ''}{' + SMOTE' if w_smote.value else ''}"
    finally:
        btn.disabled = False; btn.description = "Train"
    render()

btn.on_click(on_train)
for w in (w_thr, w_risk, w_minp, w_amt):
    w.observe(render, names="value")
display(widgets.VBox([w_model, w_cw, w_smote, btn, widgets.HTML("<b>After training, adjust:</b>"), w_thr, w_risk, w_minp, w_amt, out]))

## 11. Final model and test evaluation (once)
Rules:
1. Best configuration chosen on **validation PR-AUC** (section 5).
2. The **threshold** is chosen on validation data too: the one with the best F1.
3. Retrain that configuration on **train + validation**, with the scaler refitted on that combined set.
4. Evaluate on the **test set once**. No tuning afterwards.

In [ ]:
kind, cw, sm = CONFIGS[best_name]
print("Chosen configuration:", best_name, "->", dict(model=kind, class_weight=cw, smote=sm))

# threshold = best F1 on the validation probabilities of the train-only model
fine = np.round(np.arange(0.02, 0.99, 0.01), 2)
f1s = np.array([f1_score(y_val, (best_proba >= t).astype(int), zero_division=0) for t in fine])
FINAL_THR = float(fine[f1s.argmax()])
print(f"Threshold chosen on validation (best F1 = {f1s.max():.3f}): {FINAL_THR}")

# retrain on train + validation
final_scaler = StandardScaler().fit(X_dev[SCALE_COLS])
Xdev_s, Xtest_s = scale_with(final_scaler, X_dev), scale_with(final_scaler, X_test)
final_model = fit_model(kind, cw, sm, Xdev_s, y_dev)

# ---- the one look at the test set ----
test_proba = final_model.predict_proba(Xtest_s)[:, 1]
ts = score(y_test, test_proba, FINAL_THR)
print(f"\nTEST RESULTS (threshold {FINAL_THR})")
print(f"  Recall   : {ts['recall']*100:.1f}%   ({ts['tp']} of {ts['tp'] + ts['fn']} frauds caught)")
print(f"  Precision: {ts['precision']*100:.1f}%   ({ts['fp']} false alarms)")
print(f"  F1       : {ts['f1']:.3f}")
print(f"  PR-AUC   : {ts['pr_auc']:.3f}   ROC-AUC: {ts['roc_auc']:.3f}")
print(f"  (accuracy {ts['accuracy']*100:.2f}% - do not use this to judge the model)")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(18, 5))
ConfusionMatrixDisplay(confusion_matrix(y_test, (test_proba >= FINAL_THR).astype(int), labels=[0, 1]),
                       display_labels=["Legit", "Fraud"]).plot(ax=ax[0], cmap="Blues", values_format="d", colorbar=False)
ax[0].set_title("Confusion matrix — test")
p, r, _ = precision_recall_curve(y_test, test_proba)
ax[1].plot(r, p, lw=2); ax[1].scatter([ts["recall"]], [ts["precision"]], color="red", zorder=5, label="chosen threshold")
ax[1].set_xlabel("Recall"); ax[1].set_ylabel("Precision"); ax[1].set_title(f"PR curve — test (AP={ts['pr_auc']:.2f})"); ax[1].legend()
fpr, tpr, _ = roc_curve(y_test, test_proba)
ax[2].plot(fpr, tpr, lw=2); ax[2].plot([0, 1], [0, 1], "k--"); ax[2].set_title(f"ROC — test (AUC={ts['roc_auc']:.3f})")
ax[2].set_xlabel("False positive rate"); ax[2].set_ylabel("True positive rate")
plt.tight_layout(); plt.show()

print("Business simulation on the test set (assumptions in section 8):")
display(pd.Series(business(y_test, test_proba, FINAL_THR)).round(0).to_frame("value"))

order = np.argsort(-test_proba); yt = y_test.values
print("\nTop-K on the test set:")
display(pd.DataFrame([{"K": K, "Frauds found": int(yt[order[:K]].sum()), "Precision@K": round(yt[order[:K]].mean(), 3)} for K in (50, 100, 200)]))

alerts_test = alert_table(X_test.index, test_proba, y_test)
print("Highest-risk test transactions:")
display(alerts_test.head(10))

## 12. Time-based split check
A random split lets the model see transactions from the "future" while predicting the "past". In real life the model only ever sees the past.
Here we sort by time: first 60% train, next 20% validation (threshold choice), last 20% test, and repeat the final procedure.
Differences from section 11 show how optimistic the random split was.

In [ ]:
order_t = df["Time"].argsort().values
idx_sorted = df.index[order_t]
n = len(idx_sorted); a, b = int(n * 0.6), int(n * 0.8)
tr_i, va_i, te_i = idx_sorted[:a], idx_sorted[a:b], idx_sorted[b:]
print("Frauds in time-based train / val / test:", int(y.loc[tr_i].sum()), int(y.loc[va_i].sum()), int(y.loc[te_i].sum()))

if y.loc[te_i].sum() == 0 or y.loc[va_i].sum() == 0:
    print("A time block has no fraud, so this check cannot be run on this data.")
else:
    sc_t = StandardScaler().fit(X.loc[tr_i, SCALE_COLS])
    Xt_tr, Xt_va, Xt_te = (scale_with(sc_t, X.loc[i]) for i in (tr_i, va_i, te_i))
    m_t = fit_model(kind, cw, sm, Xt_tr, y.loc[tr_i])
    pv = m_t.predict_proba(Xt_va)[:, 1]
    f1v = np.array([f1_score(y.loc[va_i], (pv >= t).astype(int), zero_division=0) for t in fine])
    thr_t = float(fine[f1v.argmax()])
    pt = m_t.predict_proba(Xt_te)[:, 1]
    st = score(y.loc[te_i], pt, thr_t)

    cmp_t = pd.DataFrame({"Random split (section 11)": ts, "Time-based split": st}).loc[["recall", "precision", "f1", "pr_auc", "roc_auc", "tp", "fn", "fp"]].round(3)
    cmp_t.index = ["Recall", "Precision", "F1", "PR-AUC", "ROC-AUC", "Fraud caught", "Fraud missed", "False alarms"]
    display(cmp_t)
    print(f"Time-based threshold chosen on its validation block: {thr_t}")
    print("Usually the time-based numbers are a bit lower: that is the more honest estimate for the future.")

## 13. Save everything and build the Streamlit app
This writes a ready-to-deploy folder `fraud_app/`:
`app.py`, `fraud_utils.py`, `requirements.txt`, `models/best_model.pkl`, `sample_transactions.csv`.
Test it later with `streamlit run app.py`, and deploy it from GitHub on Streamlit Community Cloud.

In [ ]:
os.makedirs("outputs", exist_ok=True)
os.makedirs("fraud_app/models", exist_ok=True)

# model bundle used by the app
bundle = {"model": final_model, "scaler": final_scaler, "threshold": FINAL_THR,
          "config": {"experiment": best_name, "model": kind, "class_weight": cw, "smote": sm},
          "test_metrics": {k: float(v) for k, v in ts.items()}}
joblib.dump(bundle, "fraud_app/models/best_model.pkl", compress=3)
joblib.dump(final_scaler, "outputs/scaler.pkl")
joblib.dump(final_model, "outputs/best_model.pkl", compress=3)
with open("outputs/final_config.json", "w") as f:
    json.dump({"experiment": best_name, "threshold": FINAL_THR, "config": bundle["config"], "test_metrics": bundle["test_metrics"]}, f, indent=2)

# tables
pd.DataFrame(RESULTS).to_csv("outputs/experiment_results.csv", index=False)
th.to_csv("outputs/threshold_analysis.csv", index=False)
alerts_test.to_csv("outputs/test_alerts.csv", index=False)
alerts_test[alerts_test["Fraud probability"] >= FINAL_THR].to_csv("outputs/flagged_transactions_test.csv", index=False)

# a small demo file for the app: every test fraud + 1500 random legitimate test transactions
rng = np.random.RandomState(SEED)
fraud_idx = y_test.index[y_test == 1]
legit_idx = rng.choice(y_test.index[y_test == 0], size=min(1500, int((y_test == 0).sum())), replace=False)
df.loc[np.concatenate([fraud_idx, legit_idx])].sample(frac=1, random_state=SEED).to_csv("fraud_app/sample_transactions.csv", index=False)

# figures
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
for nme in names_ex_dummy:
    p_, r_, _ = precision_recall_curve(y_val, PROBAS[nme]); ax[0].plot(r_, p_, label=nme)
    f_, t_, _ = roc_curve(y_val, PROBAS[nme]); ax[1].plot(f_, t_, label=nme)
ax[0].set_title("PR curves (validation)"); ax[0].legend(fontsize=7); ax[1].plot([0, 1], [0, 1], "k--"); ax[1].set_title("ROC curves (validation)")
plt.tight_layout(); plt.savefig("outputs/pr_roc_curves.png", dpi=150); plt.close()
ConfusionMatrixDisplay(confusion_matrix(y_test, (test_proba >= FINAL_THR).astype(int), labels=[0, 1]),
                       display_labels=["Legit", "Fraud"]).plot(cmap="Blues", values_format="d")
plt.title("Confusion matrix — test"); plt.savefig("outputs/confusion_matrix_test.png", dpi=150); plt.close()
plt.figure(figsize=(7, 5)); plt.plot(th["Threshold"], th["Precision"], label="Precision"); plt.plot(th["Threshold"], th["Recall"], label="Recall")
plt.plot(th["Threshold"], th["F1"], label="F1"); plt.legend(); plt.title("Metrics vs threshold (validation)"); plt.savefig("outputs/threshold_curve.png", dpi=150); plt.close()
print("Saved to outputs/ and fraud_app/")

In [ ]:
APP_SRC = """
import json
import joblib
import numpy as np
import pandas as pd
import streamlit as st
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, average_precision_score

from fraud_utils import prepare, risk_level, validate, RISK_LABELS

st.set_page_config(page_title="Fraud Risk Monitor", page_icon="💳", layout="wide")


@st.cache_resource
def load_bundle():
    return joblib.load("models/best_model.pkl")


bundle = load_bundle()
model, scaler = bundle["model"], bundle["scaler"]

st.title("💳 Credit Card Fraud Risk Monitor")
st.caption("Educational demo. Scores are model estimates, not real bank decisions.")

with st.sidebar:
    st.header("Data")
    uploaded = st.file_uploader("Upload transactions (CSV, same columns as creditcard.csv)", type="csv")
    st.header("Detection")
    threshold = st.slider("Fraud threshold", 0.01, 0.99, float(bundle["threshold"]), 0.01)
    st.header("Business assumptions")
    avg_loss = st.number_input("Average fraud loss ($)", 0.0, 100000.0, 200.0, 10.0)
    alert_cost = st.number_input("Cost per investigated alert ($)", 0.0, 1000.0, 5.0, 1.0)
    prevent = st.slider("Share of detected fraud prevented", 0.0, 1.0, 0.8, 0.05)

if uploaded is not None:
    raw = pd.read_csv(uploaded)
    source = "your upload"
else:
    raw = pd.read_csv("sample_transactions.csv")
    source = "the built-in demo sample"

problems = validate(raw)
if problems:
    st.error("Cannot score this file: " + "; ".join(problems))
    st.stop()

X = prepare(raw, scaler)
proba = model.predict_proba(X)[:, 1]
out = pd.DataFrame({"Transaction": raw.index, "Amount ($)": raw["Amount"].round(2).values,
                    "Fraud probability": np.round(proba, 4)})
out["Risk"] = risk_level(out["Fraud probability"]).astype(str).values
out["Decision"] = np.where(out["Fraud probability"] >= threshold, "Flag for review", "Allow")
has_labels = "Class" in raw.columns
if has_labels:
    out["Actually fraud"] = np.where(raw["Class"].values == 1, "YES", "no")

flagged = out[out["Decision"] == "Flag for review"]
st.subheader(f"Results for {source}")
c1, c2, c3, c4 = st.columns(4)
c1.metric("Transactions", f"{len(out):,}")
c2.metric("Flagged", f"{len(flagged):,}")
c3.metric("Flagged amount", f"${flagged['Amount ($)'].sum():,.0f}")
c4.metric("Critical risk", f"{(out['Risk'] == 'Critical').sum():,}")

if has_labels:
    y = raw["Class"].values
    pred = (proba >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    st.markdown("**Performance on the labelled data**")
    m1, m2, m3, m4 = st.columns(4)
    m1.metric("Recall", f"{recall_score(y, pred, zero_division=0):.1%}", f"{tp} of {tp + fn} frauds caught")
    m2.metric("Precision", f"{precision_score(y, pred, zero_division=0):.1%}", f"{fp} false alarms")
    m3.metric("F1", f"{f1_score(y, pred, zero_division=0):.3f}")
    m4.metric("PR-AUC", f"{average_precision_score(y, proba):.3f}" if y.sum() else "n/a")
    prevented = tp * avg_loss * prevent
    cost = len(flagged) * alert_cost
    st.markdown("**Business simulation (assumptions in the sidebar)**")
    b1, b2, b3 = st.columns(3)
    b1.metric("Losses prevented", f"${prevented:,.0f}")
    b2.metric("Investigation cost", f"${cost:,.0f}")
    b3.metric("Net benefit", f"${prevented - cost:,.0f}")

st.subheader("Fraud alert list")
f1c, f2c = st.columns(2)
risks = f1c.multiselect("Risk levels", RISK_LABELS, default=RISK_LABELS)
min_p = f2c.slider("Minimum probability", 0.0, 1.0, 0.0, 0.05)
view = out[out["Risk"].isin(risks) & (out["Fraud probability"] >= min_p)].sort_values("Fraud probability", ascending=False)
st.dataframe(view.head(500), width="stretch", hide_index=True)

st.download_button("Download flagged transactions (CSV)", flagged.sort_values("Fraud probability", ascending=False).to_csv(index=False).encode(),
                   file_name="flagged_transactions.csv", mime="text/csv")
"""
with open("fraud_app/app.py", "w") as f:
    f.write(APP_SRC)
shutil.copy("fraud_utils.py", "fraud_app/fraud_utils.py")
with open("fraud_app/requirements.txt", "w") as f:
    f.write("streamlit>=1.50\nscikit-learn\npandas\nnumpy\njoblib\n" + ("xgboost\n" if kind == "xgb" else ""))
with open("fraud_app/README.md", "w") as f:
    f.write("# Fraud Risk Monitor\n\nRun locally:\n\n    pip install -r requirements.txt\n    streamlit run app.py\n\nDeploy: push this folder to GitHub, then create an app on share.streamlit.io pointing at app.py.\n")

print("fraud_app/ contents:")
for root, _, files_ in os.walk("fraud_app"):
    for fn in files_:
        p = os.path.join(root, fn); print(f"  {p}  ({os.path.getsize(p)/1e6:.1f} MB)")

In [ ]:
# One zip with everything (Colab downloads it). Elsewhere it just stays in the current folder.
os.makedirs("fraud_project", exist_ok=True)
shutil.copytree("outputs", "fraud_project/outputs", dirs_exist_ok=True)
shutil.copytree("fraud_app", "fraud_project/fraud_app", dirs_exist_ok=True)
shutil.make_archive("fraud_project", "zip", "fraud_project")
try:
    from google.colab import files
    files.download("fraud_project.zip")
except ImportError:
    print("Not in Colab: fraud_project.zip is in the current folder.")

## Next steps
1. Fill in the EDA summary (3.6) and your answers to the report questions (section 6).
2. Test the app: download the zip, unzip, then `cd fraud_app`, `pip install -r requirements.txt`, `streamlit run app.py`.
3. Put `fraud_app/` on GitHub and deploy on Streamlit Community Cloud. The model file is within GitHub's 100 MB limit when FAST_MODE is on;
   if you set it to False and the file is larger, tell me and I'll show you how to shrink it.
4. Write the README and make slides: problem, imbalance, models, thresholds, business value, limits.

**Limitations to mention:** anonymised PCA features, only two days of data from European cardholders (2013), the dollar figures are assumptions, and fraud patterns change over time.